# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [ ]:
!pip install -q langchain-community sentence-transformers chromadb

In [3]:
# YOUR CODE HERE
import glob
import os
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Load SOP documents (.md files) from corporate_policies/
sop_files = sorted(glob.glob("corporate_policies/*.md"))
docs = []
for file_path in sop_files:
    loader = TextLoader(file_path, encoding="utf-8")
    docs.extend(loader.load())

print(f"Loaded {len(docs)} SOP documents.")

# 2. Initialize the all-MiniLM-L6-v2 embedding model
embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

# 3. Validate by embedding a sample string and checking vector length (384 dims)
sample_vec = embeddings.embed_query("Testing embedding vector dimensions for MiniLM.")
print("Sample vector length:", len(sample_vec))
assert len(sample_vec) == 384, f"Expected 384 dimensions, got {len(sample_vec)}"

# 4. Embed all SOP documents to validate they embed cleanly
doc_vectors = embeddings.embed_documents([doc.page_content for doc in docs])
print(f"Successfully generated {len(doc_vectors)} document embeddings of dimension {len(doc_vectors[0])}.")

/tmp/ipykernel_533/3742189067.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Loaded 5 SOP documents.


/tmp/ipykernel_533/3742189067.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

... (7 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Sample vector length: 384
Successfully generated 5 document embeddings of dimension 384.


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [4]:
# YOUR CODE HERE
from langchain_community.vectorstores import Chroma

# 1. Create a persistent Chroma vector index from the loaded SOP documents
persist_directory = "./chroma_db"
vector_db = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory=persist_directory
)

# 2. Validate that the index count matches the number of SOP documents
doc_count = vector_db._collection.count()
print("Stored vectors in ChromaDB:", doc_count)
assert doc_count == len(docs), f"Expected {len(docs)} vectors in ChromaDB, got {doc_count}"

# 3. Run a test similarity search for "refund"
test_results = vector_db.similarity_search("refund", k=1)
print("\nTest similarity search for 'refund' (top-1):")
print("Source metadata:", test_results[0].metadata)
print("Content preview:", test_results[0].page_content[:250])

Stored vectors in ChromaDB: 5

Test similarity search for 'refund' (top-1):
Source metadata: {'source': 'corporate_policies/refund_policy.md'}
Content preview: # Refund Policy

## Eligibility
Customers may request a refund within 30 days of the original purchase or
delivery date, whichever is later. To be eligible, the item must be unused or
defective, and the request must reference a valid order identifier


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [5]:
# YOUR CODE HERE
import json
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load test_query (and baseline info) from outputs.json
with open("outputs.json", "r", encoding="utf-8") as f:
    artifacts = json.load(f)

test_query = artifacts.get(
    "test_query",
    "I was charged twice for my subscription and my account is locked out when I try to check the invoice!"
)
print("Customer Query:", test_query)

# 2. Naive retrieval: fetch the top-1 matching SOP from ChromaDB
retrieved_docs = vector_db.similarity_search(test_query, k=1)
top_doc = retrieved_docs[0]
retrieved_source = top_doc.metadata.get("source", "unknown")
retrieved_context = top_doc.page_content

print("Retrieved Document:", retrieved_source)
print("Retrieved Snippet:\n", retrieved_context[:300], "...\n")

# 3. Load the Qwen base model to generate the Naive RAG response
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

# 4. Inject the retrieved SOP into the system prompt
system_prompt = f"Answer strictly using this SOP: {retrieved_context}"
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_query}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=256,
        do_sample=False
    )

# Decode only the newly generated tokens
generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]
naive_rag_output = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

print("=== Naive RAG Output ===")
print(naive_rag_output)

Customer Query: My order hasn't arrived yet. How long does domestic shipping usually take and how do I track it?
Retrieved Document: corporate_policies/shipping_delays.md
Retrieved Snippet:
 # Shipping Delays

## Scope
This procedure covers orders that have shipped but are running behind the
estimated delivery date, as well as orders stuck in a pre-shipment state.

## Standard Timelines
Domestic orders deliver within 3–7 business days; international orders within
10–21 business days. An ...



config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

=== Naive RAG Output ===
Domestic shipping typically takes around 3-7 business days. To track your order, you should check the carrier's website or use the tracking number provided with your invoice. This will show you the current status of your package, including any updates from the carrier. If there are any issues or concerns, please contact our customer service team for assistance.


---
## Save Artifacts for Downstream Notebooks

In [6]:
# YOUR CODE HERE
import json
import os

# 1. Verify ./chroma_db exists on disk
persist_directory = "./chroma_db"
assert os.path.exists(persist_directory), f"Missing {persist_directory} directory on disk!"
print("Verified ./chroma_db exists at:", os.path.abspath(persist_directory))

# 2. Load existing outputs.json (or start with a dict if not present)
outputs_path = "outputs.json"
if os.path.exists(outputs_path):
    with open(outputs_path, "r", encoding="utf-8") as f:
        data = json.load(f)
else:
    data = {}

# 3. Update outputs.json with Naive RAG metadata and generated output
data["naive_rag_retrieved_source"] = retrieved_source
data["naive_rag_output"] = naive_rag_output

with open(outputs_path, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print("Updated outputs.json with keys:", list(data.keys()))

Verified ./chroma_db exists at: /content/chroma_db
Updated outputs.json with keys: ['test_query', 'ground_truth', 'baseline_output', 'hallucination_notes', 'naive_rag_retrieved_source', 'naive_rag_output']


In [8]:
!zip -r chroma_db.zip chroma_db

  adding: chroma_db/ (stored 0%)
  adding: chroma_db/chroma.sqlite3 (deflated 81%)
  adding: chroma_db/acdb434e-4e81-4c8a-95f1-07b75df419dc/ (stored 0%)
  adding: chroma_db/acdb434e-4e81-4c8a-95f1-07b75df419dc/header.bin (deflated 63%)
  adding: chroma_db/acdb434e-4e81-4c8a-95f1-07b75df419dc/length.bin (deflated 63%)
  adding: chroma_db/acdb434e-4e81-4c8a-95f1-07b75df419dc/link_lists.bin (stored 0%)
  adding: chroma_db/acdb434e-4e81-4c8a-95f1-07b75df419dc/data_level0.bin (deflated 100%)


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**